# Historical model comparison

This notebook records the comparison that led to the legacy XGBoost settings in
`train.py`. The reported results use random splits and folds after legacy
cleaning and asking-price filters. Validation rows are trimmed using observed
asking rent per square metre and its ratio to the training-derived local market
rate. Deduplication also uses asking price.

The historical procedure was to load the cleaned data, use an 80/20 random split,
compare the candidate models in euros, then run five random folds and tune the
leading candidates. The recorded scores describe the listings retained in those
splits and folds.

A14's benchmark implementation is deferred to M2. The
[A14 handoff](../docs/tasks/2026-11-05-evaluation-handoff/plan.md#m2-handoff)
records timestamped observation retention, immutable dataset releases, reviewed
property identities, and common evaluation membership. The frozen
[evaluation protocol](../docs/architecture/specification-v1.md#95-dual-evaluation-protocol)
requires future-market evaluation and a secondary unseen-property view.

In [ ]:
import sys

sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot

from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

from features import (clean_data, build_ppm_grid, assign_ppm, trim_ppm_outliers,
                      trim_knn_outliers, impute_medians, FEATURES)

df = pd.read_csv("../data/raw/listings.csv")
df = clean_data(df)
print(len(df), "clean listings")
df[["price", "area", "rooms", "dist_to_center"]].describe()

In [ ]:
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = df_train.copy(), df_val.copy()

# same leak-free preprocessing train.py does: the 1% price/m2 trim and the
# median imputation are FIT on train only, then applied to the validation rows.
df_train, ppm_bounds = trim_ppm_outliers(df_train)
df_val, _ = trim_ppm_outliers(df_val, ppm_bounds)
df_train, medians = impute_medians(df_train)
df_val, _ = impute_medians(df_val, medians)

grid_ppm, sector_ppm, global_ppm = build_ppm_grid(df_train)
df_train['knn_ppm'] = assign_ppm(df_train, grid_ppm, sector_ppm, global_ppm)
df_val['knn_ppm'] = assign_ppm(df_val, grid_ppm, sector_ppm, global_ppm)

# and the same market-rate trim train.py does: drop listings asking way off
# their own neighborhood, then recompute the rate from the cleaned market
df_train = trim_knn_outliers(df_train).copy()
df_val = trim_knn_outliers(df_val).copy()
grid_ppm, sector_ppm, global_ppm = build_ppm_grid(df_train)
df_train['knn_ppm'] = assign_ppm(df_train, grid_ppm, sector_ppm, global_ppm)
df_val['knn_ppm'] = assign_ppm(df_val, grid_ppm, sector_ppm, global_ppm)

model_features = FEATURES + ['knn_ppm']

y_train = np.log1p(df_train['price'].values)
y_val = df_val['price'].values

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[model_features].to_dict(orient='records'))
X_val = dv.transform(df_val[model_features].to_dict(orient='records'))

(X_train.shape, X_val.shape)

### The candidates

From dumbest to fanciest, so every better model has to justify itself against the one below it:

- **median EUR/m2 × area** - the dumbest thing that could possibly work. One price per square meter for the whole city. This is the floor everyone must beat.
- **linear regression / ridge** - cheap, linear, interpretable. How far does a straight line get us?
- **random forest, gradient boosting** (sklearn) - the tree ensembles.
- **xgboost** - the model `train.py` actually uses, and the one on trial here.

In [ ]:
def score(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE (%)": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        "R2": r2_score(y_true, y_pred),
    }

predictions = {}

In [ ]:
ppm_median = (df_train.price / df_train.area).median()
predictions['median EUR/m2 * area'] = ppm_median * df_val['area'].values
print(f'the dumbest of the dumbest charges {ppm_median:.2f} EUR/m2 for everything')

In [ ]:
xgb_params = dict(n_estimators=500, max_depth=5, learning_rate=0.1,
                  subsample=0.8, colsample_bytree=0.9, min_child_weight=8,
                  random_state=42) # picked some random values, we'll see best ones soon

models = {
    "linear regression": LinearRegression(),
    "ridge": Ridge(alpha=1.0),
    "random forest": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=1, max_features=0.5, random_state=42, n_jobs=-1),
    "gradient boosting": GradientBoostingRegressor(random_state=42),
    "xgboost": XGBRegressor(**xgb_params),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = np.expm1(model.predict(X_val))

results = pd.DataFrame({name: score(y_val, p) for name, p in predictions.items()}).T
results.sort_values("RMSE").round(2)

In the recorded filtered random split, the tree models had RMSE near
190 EUR, compared with about 290 EUR for the baseline. Linear regression explained
about 60% of variance in that retained validation population.

XGBoost's reported RMSE was about 3 EUR below random forest on that split.
The next cells repeat the comparison over five random folds before tuning.

In [ ]:
def cross_valid_rmse(make_model, n_splits=5):
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    rmses = []

    for train_idx, val_idx in kfold.split(df):
        d_tr, d_va = df.iloc[train_idx].copy(), df.iloc[val_idx].copy()

        # fit the trim + imputation on this fold's train rows only, then apply
        # to the held-out rows - no validation row leaks into the preprocessing
        d_tr, bounds = trim_ppm_outliers(d_tr)
        d_va, _ = trim_ppm_outliers(d_va, bounds)
        d_tr, medians = impute_medians(d_tr)
        d_va, _ = impute_medians(d_va, medians)

        grid_ppm, sector_ppm, global_ppm = build_ppm_grid(d_tr)
        d_tr['knn_ppm'] = assign_ppm(d_tr, grid_ppm, sector_ppm, global_ppm)
        d_va['knn_ppm'] = assign_ppm(d_va, grid_ppm, sector_ppm, global_ppm)

        # the market-rate trim, exactly like train.py: fixed 0.5x-2x band,
        # then the neighbor rate gets recomputed from the trimmed train rows
        d_tr = trim_knn_outliers(d_tr).copy()
        d_va = trim_knn_outliers(d_va).copy()
        grid_ppm, sector_ppm, global_ppm = build_ppm_grid(d_tr)
        d_tr['knn_ppm'] = assign_ppm(d_tr, grid_ppm, sector_ppm, global_ppm)
        d_va['knn_ppm'] = assign_ppm(d_va, grid_ppm, sector_ppm, global_ppm)
        
        dv_ = DictVectorizer(sparse=False)
        X_tr = dv_.fit_transform(d_tr[model_features].to_dict(orient='records'))
        X_va = dv_.transform(d_va[model_features].to_dict(orient='records'))
        m = make_model()
        
        m.fit(X_tr, np.log1p(d_tr["price"].values))
        cv_preds = np.expm1(m.predict(X_va))
        rmses.append(np.sqrt(mean_squared_error(d_va["price"].values, cv_preds)))
        
    return np.mean(rmses), np.std(rmses)

factories = {
    "linear regression": lambda: LinearRegression(),
    "random forest": lambda: RandomForestRegressor(
        n_estimators=300, min_samples_leaf=1, max_features=0.5, random_state=42, n_jobs=-1),
    "gradient boosting": lambda: GradientBoostingRegressor(random_state=42),
    "xgboost": lambda: XGBRegressor(**xgb_params),
}

for name, factory in factories.items():
    mean, std = cross_valid_rmse(factory)
    print(f"{name}  RMSE {mean:4.0f} +- {std:.0f} EUR")

The historical five random folds reported XGBoost RMSE of **198 +- 6 EUR** and
random forest RMSE of **200 +- 5 EUR** before tuning. These summaries apply to
the filtered validation rows in those folds. The next cells record the
historical searches for both models.

In [ ]:
rng = np.random.default_rng(42)

def random_xgb_config():
    return {
        "n_estimators": int(rng.choice([200, 300, 400, 600, 800])),
        "max_depth": int(rng.integers(3, 9)),
        "learning_rate": float(rng.choice([0.02, 0.03, 0.05, 0.08, 0.12])),
        "subsample": round(float(rng.uniform(0.7, 1.0)), 2),
        "colsample_bytree": round(float(rng.uniform(0.7, 1.0)), 2),
        "min_child_weight": int(rng.integers(1, 9)),
    }

xgb_results = []
for i in range(10):
    cfg = random_xgb_config()
    mean, std = cross_valid_rmse(lambda: XGBRegressor(random_state=42, **cfg))
    xgb_results.append({**cfg, "cv_rmse": round(mean, 1), "std": round(std, 1)})
    print(f"{i + 1:2d}/10  RMSE {mean:5.1f} +- {std:4.1f}  {cfg}")

pd.DataFrame(xgb_results).sort_values("cv_rmse").head(5)

### Tuning random forest too, to keep it fair

It'd be cheap to tune only XGBoost and declare victory. Since random forest was the one breathing down its neck, it gets a proper search as well - otherwise the comparison isn't honest.

In [ ]:
rf_grid = [
    {"n_estimators": n, "min_samples_leaf": leaf, "max_features": mf}
    for n in [300] for leaf in [1, 3, 5] for mf in [0.5, 1.0]
]

rf_results = []
for cfg in rf_grid:
    mean, std = cross_valid_rmse(lambda: RandomForestRegressor(random_state=42, n_jobs=-1, **cfg))
    rf_results.append({**cfg, "cv_rmse": round(mean, 1), "std": round(std, 1)})
    print(f"RMSE {mean:5.1f} +- {std:4.1f}  {cfg}")

pd.DataFrame(rf_results).sort_values("cv_rmse")

### Historical tuning results

The historical searches reported the following random-fold CV results.

| model | untuned CV RMSE | best tuned CV RMSE |
|---|---|---|
| xgboost | 198 +- 6 | 192.4 +- 6.2 |
| random forest | 200 +- 5 | 199.8 +- 5.0 |

- XGBoost dropped from 198 to 192.4 EUR. The best draw was `n_estimators=600, max_depth=7, learning_rate=0.08, subsample=0.84, colsample_bytree=0.71, min_child_weight=2`, but the config `train.py` actually ships (`600, max_depth=6, lr=0.05, subsample=0.83, colsample_bytree=0.77, min_child_weight=7`) had a reported CV RMSE of 192.5 EUR on the filtered historical data.
- Random forest basically didn't move: its default-ish config (`min_samples_leaf=1, max_features=0.5`) was already the best of the grid at 199.8, and every other tested setting had higher reported CV RMSE.

These historical results supported the legacy XGBoost settings. The random search used ten draws. Future candidate selection will use the M2 dataset and split manifest, with later observations, reviewed property identities, and a common evaluation population.